In [1]:
import pandas as pd

train = pd.read_csv('train_cleaned.csv')
train['StateHoliday'] = train['StateHoliday'].astype(str)

open_days = train[train['Open'] == 1]

promo_summary = open_days.groupby('Promo')['Sales'].agg(['mean', 'median', 'std', 'count'])
promo_summary

/tmp/ipykernel_1031/3759407754.py:3: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv('train_cleaned.csv')


,mean,median,std,count
Promo,,,,
0,5929.407603,5459.0,2629.648385,467496
1,8228.281239,7649.0,3175.759157,376896


### Promo and Sales: First Look

**Question:** Are sales higher when a promotion is active?

**Method:** Compared sales on promo and non-promo days, using only rows where the store was open.

**Result:** Average sales were 8,228 on promo days versus 5,929 on non-promo days — a raw difference of 2,299 (38.8%). Median sales were also higher on promo days.

**Takeaway:** Promotions are strongly associated with higher sales. However, this first comparison pools all stores and does not yet control for store size, weekday, holidays, or seasonality.

In [2]:
promo_by_store = (open_days.groupby(['Store', 'Promo'])['Sales'].agg(['mean', 'median', 'count']).reset_index())

promo_by_store

,Store,Promo,mean,median,count
0,1,0,4319.756381,4086.0,431
1,1,1,5300.111429,5210.0,350
2,2,0,3864.288372,3924.0,430
3,2,1,6277.440678,6281.0,354
4,3,0,5382.613054,5333.0,429
...,...,...,...,...,...
2225,1113,1,7465.243626,7203.0,353
2226,1114,0,19549.990719,19129.0,431
2227,1114,1,22029.855524,21498.0,353
2228,1115,0,5512.419954,5245.0,431


In [3]:
promo_pivot = (promo_by_store.pivot(
        index='Store',
        columns='Promo',
        values=['mean', 'median', 'count']
    )
)

promo_pivot

mean                 median           count       
Promo             0             1        0        1      0      1
Store                                                            
1       4319.756381   5300.111429   4086.0   5210.0  431.0  350.0
2       3864.288372   6277.440678   3924.0   6281.0  430.0  354.0
3       5382.613054   8854.628571   5333.0   8612.5  429.0  350.0
4       8870.354988  10576.158640   8550.0  10157.0  431.0  353.0
5       3511.406542   6096.683761   3717.0   5884.0  428.0  351.0
...             ...           ...      ...      ...    ...    ...
1111    4169.627040   6578.017143   4157.0   6279.0  429.0  350.0
1112    8106.673660  12847.231429   7934.0  12346.0  429.0  350.0
1113    5942.020882   7465.243626   5667.0   7203.0  431.0  353.0
1114   19549.990719  22029.855524  19129.0  21498.0  431.0  353.0
1115    5512.419954   7275.360000   5245.0   7033.0  431.0  350.0

[1115 rows x 6 columns]

In [4]:
promo_pivot.columns = [
    f'{metric}_promo_{promo_status}'
    for metric, promo_status in promo_pivot.columns
]

promo_pivot

,mean_promo_0,mean_promo_1,median_promo_0,median_promo_1,count_promo_0,count_promo_1
Store,,,,,,
1,4319.756381,5300.111429,4086.0,5210.0,431.0,350.0
2,3864.288372,6277.440678,3924.0,6281.0,430.0,354.0
3,5382.613054,8854.628571,5333.0,8612.5,429.0,350.0
4,8870.354988,10576.158640,8550.0,10157.0,431.0,353.0
5,3511.406542,6096.683761,3717.0,5884.0,428.0,351.0
...,...,...,...,...,...,...
1111,4169.627040,6578.017143,4157.0,6279.0,429.0,350.0
1112,8106.673660,12847.231429,7934.0,12346.0,429.0,350.0
1113,5942.020882,7465.243626,5667.0,7203.0,431.0,353.0


In [5]:
promo_pivot = promo_pivot.dropna(subset=['mean_promo_0', 'mean_promo_1']).copy()

In [6]:
promo_pivot['absolute_lift'] = (promo_pivot['mean_promo_1'] - promo_pivot['mean_promo_0'])

In [7]:
promo_pivot['pct_lift'] = (promo_pivot['absolute_lift']/ promo_pivot['mean_promo_0']* 100)

promo_pivot

,mean_promo_0,mean_promo_1,median_promo_0,median_promo_1,count_promo_0,count_promo_1,absolute_lift,pct_lift
Store,,,,,,,,
1,4319.756381,5300.111429,4086.0,5210.0,431.0,350.0,980.355048,22.694684
2,3864.288372,6277.440678,3924.0,6281.0,430.0,354.0,2413.152306,62.447521
3,5382.613054,8854.628571,5333.0,8612.5,429.0,350.0,3472.015518,64.504275
4,8870.354988,10576.158640,8550.0,10157.0,431.0,353.0,1705.803652,19.230388
5,3511.406542,6096.683761,3717.0,5884.0,428.0,351.0,2585.277219,73.625118
...,...,...,...,...,...,...,...,...
1111,4169.627040,6578.017143,4157.0,6279.0,429.0,350.0,2408.390103,57.760324
1112,8106.673660,12847.231429,7934.0,12346.0,429.0,350.0,4740.557769,58.477225
1113,5942.020882,7465.243626,5667.0,7203.0,431.0,353.0,1523.222744,25.634759


In [8]:
promo_pivot[
    [
        'mean_promo_0',
        'mean_promo_1',
        'count_promo_0',
        'count_promo_1',
        'absolute_lift',
        'pct_lift'
    ]
].describe().round(2)

,mean_promo_0,mean_promo_1,count_promo_0,count_promo_1,absolute_lift,pct_lift
count,1115.00,1115.00,1115.00,1115.00,1115.00,1115.00
mean,5906.57,8205.18,419.28,338.02,2298.60,41.43
std,2197.29,2710.16,37.95,28.51,1023.79,18.10
min,1753.46,3242.84,329.00,263.00,-290.55,-6.95
25%,4465.08,6384.72,428.00,348.00,1572.23,29.36
50%,5567.84,7854.21,429.00,350.00,2192.78,40.56
75%,6828.59,9435.02,431.00,352.50,2857.20,51.84
max,20549.64,25168.96,582.00,360.00,7868.75,144.93


### Promo Effect Within Each Store

**Method:** For each store, compared its average sales on promo days with its own average sales on non-promo days.

**Result:** All 1,115 stores had both types of days. The median store had 2,193 higher daily sales on promo days, equal to a 40.6% increase. The middle 50% of stores had promo lifts between 29.4% and 51.8%.

**Takeaway:** The promo-sales relationship remains strong after controlling for each store's normal sales level. Promo impact is widespread, but the size of the lift varies across stores.

In [9]:
store_type_map = (open_days[['Store', 'StoreType']].drop_duplicates('Store'))

store_type_map

,Store,StoreType
0,1,c
1,2,a
2,3,a
3,4,c
4,5,a
...,...,...
1112,1113,a
1113,1114,a
1114,1115,d
18715,876,a


In [10]:
promo_by_store_type = (promo_pivot.reset_index().merge(store_type_map, on='Store', how='left'))

promo_by_store_type

,Store,mean_promo_0,mean_promo_1,median_promo_0,median_promo_1,count_promo_0,count_promo_1,absolute_lift,pct_lift,StoreType
0,1,4319.756381,5300.111429,4086.0,5210.0,431.0,350.0,980.355048,22.694684,c
1,2,3864.288372,6277.440678,3924.0,6281.0,430.0,354.0,2413.152306,62.447521,a
2,3,5382.613054,8854.628571,5333.0,8612.5,429.0,350.0,3472.015518,64.504275,a
3,4,8870.354988,10576.158640,8550.0,10157.0,431.0,353.0,1705.803652,19.230388,c
4,5,3511.406542,6096.683761,3717.0,5884.0,428.0,351.0,2585.277219,73.625118,a
...,...,...,...,...,...,...,...,...,...,...
1110,1111,4169.627040,6578.017143,4157.0,6279.0,429.0,350.0,2408.390103,57.760324,a
1111,1112,8106.673660,12847.231429,7934.0,12346.0,429.0,350.0,4740.557769,58.477225,c
1112,1113,5942.020882,7465.243626,5667.0,7203.0,431.0,353.0,1523.222744,25.634759,a
1113,1114,19549.990719,22029.855524,19129.0,21498.0,431.0,353.0,2479.864805,12.684736,a


In [11]:
storetype_promo_summary = (promo_by_store_type.groupby('StoreType')['pct_lift'].agg(['count', 'mean', 'median', 'std', 'min', 'max']).round(2))

print(storetype_promo_summary)

           count   mean  median    std   min     max
StoreType                                           
a            602  45.82   43.96  18.92  3.24  144.93
b             17  19.09   10.25  23.79 -6.95   76.35
c            148  35.46   34.02  16.12  3.53  110.16
d            348  37.48   36.50  14.38  9.10   98.95


In [12]:
promo_by_store_type[promo_by_store_type['StoreType'] == 'b'][
    ['Store', 'pct_lift', 'mean_promo_0', 'mean_promo_1', 'count_promo_0', 'count_promo_1']
].sort_values('pct_lift')

,Store,pct_lift,mean_promo_0,mean_promo_1,count_promo_0,count_promo_1
273,274,-6.946978,4182.435540,3891.882682,574.0,358.0
261,262,2.150396,20549.637457,20991.536111,582.0,360.0
947,948,2.707427,6865.652705,7051.535211,573.0,355.0
352,353,4.492902,5549.949565,5799.303371,575.0,356.0
675,676,5.543907,7529.674256,7947.112392,571.0,347.0
84,85,6.332688,7100.609966,7550.269444,582.0,360.0
258,259,7.663852,11311.573913,12178.476190,575.0,357.0
422,423,9.379801,10453.676976,11434.211111,582.0,360.0
732,733,10.251886,14370.273196,15843.497222,582.0,360.0
511,512,10.499526,5301.045259,5857.629893,464.0,281.0


In [13]:
assortment_map = open_days[['Store', 'Assortment']].drop_duplicates('Store')

promo_by_store_type_assort = promo_by_store_type.merge(assortment_map, on='Store', how='left')

pd.crosstab(promo_by_store_type_assort['StoreType'], promo_by_store_type_assort['Assortment'])

Assortment,a,b,c
StoreType,,,
a,381,0,221
b,7,9,1
c,77,0,71
d,128,0,220


### Does Promo Lift Differ by StoreType?

**Question:** Does promo effect vary by store format?

**Method:** Compared per-store promo lift (from the earlier analysis) grouped by StoreType. Checked StoreType 'b' individually since it only has 17 stores, and cross-tabulated StoreType against Assortment to check for confounding.

**Result:** StoreType 'a' shows the strongest, most consistent lift (median 43.96%). StoreType b's low median (10.25%) hides a real split: some stores show near-zero lift, others show 56–76% lift. Assortment 'b' appears almost exclusively within StoreType b, so the two variables are confounded for this group.

**Takeaway:** StoreType does seem to relate to promo responsiveness, but for StoreType 'b' specifically we cannot separate the effect of store format from assortment type given the small sample and near-total overlap.

---

Business question: Does having a closer competitor actually hurt a store's sales, or is any apparent relationship just a byproduct of which stores happen to have close competitors (e.g., store format, urban density)?

In [14]:
store_level = (
    open_days
    .groupby('Store')
    .agg(
        avg_sales=('Sales', 'mean'),
        median_sales=('Sales', 'median'),
        StoreType=('StoreType', 'first'),
        Assortment=('Assortment', 'first'),
        CompetitionDistance=('CompetitionDistance', 'first')
    )
    .reset_index()
)

store_level

,Store,avg_sales,median_sales,StoreType,Assortment,CompetitionDistance
0,1,4759.096031,4647.0,c,a,1270.0
1,2,4953.900510,4783.0,a,a,570.0
2,3,6942.568678,6619.0,a,a,14130.0
3,4,9638.401786,9430.5,c,c,620.0
4,5,4676.274711,4616.0,a,a,29910.0
...,...,...,...,...,...,...
1110,1111,5251.702182,5028.0,a,a,1900.0
1111,1112,10236.577664,9410.0,c,c,1880.0
1112,1113,6627.859694,6354.5,a,c,9260.0
1113,1114,20666.562500,20412.5,a,c,870.0


In [15]:
store_level['CompetitionDistance'].describe()

,CompetitionDistance
count,1112.000000
mean,5404.901079
std,7663.174720
min,20.000000
25%,717.500000
50%,2325.000000
75%,6882.500000
max,75860.000000


In [17]:
store_level["CompetitionDistance"].isnull().sum()

np.int64(3)

In [18]:
import numpy as np
from scipy import stats

store_level_clean = store_level.dropna(subset=['CompetitionDistance']).copy()

store_level_clean['log_CompetitionDistance'] = np.log1p(store_level_clean['CompetitionDistance'])

In [19]:
store_level_clean

,Store,avg_sales,median_sales,StoreType,Assortment,CompetitionDistance,log_CompetitionDistance
0,1,4759.096031,4647.0,c,a,1270.0,7.147559
1,2,4953.900510,4783.0,a,a,570.0,6.347389
2,3,6942.568678,6619.0,a,a,14130.0,9.556126
3,4,9638.401786,9430.5,c,c,620.0,6.431331
4,5,4676.274711,4616.0,a,a,29910.0,10.305982
...,...,...,...,...,...,...,...
1110,1111,5251.702182,5028.0,a,a,1900.0,7.550135
1111,1112,10236.577664,9410.0,c,c,1880.0,7.539559
1112,1113,6627.859694,6354.5,a,c,9260.0,9.133567
1113,1114,20666.562500,20412.5,a,c,870.0,6.769642


In [20]:
pearson_raw, p_pearson_raw = stats.pearsonr(store_level_clean['CompetitionDistance'], store_level_clean['avg_sales'])

In [21]:
spearman_raw, p_spearman_raw = stats.spearmanr(store_level_clean['CompetitionDistance'], store_level_clean['avg_sales'])

In [22]:
pearson_log, p_pearson_log = stats.pearsonr(store_level_clean['log_CompetitionDistance'], store_level_clean['avg_sales'])

In [26]:
print(f"Pearson (raw distance):   r={pearson_raw:.4f}, p={p_pearson_raw:.4g}")

print(f"Spearman (raw distance):  rho={spearman_raw:.4f}, p={p_spearman_raw:.4g}")

print(f"Pearson (log distance):   r={pearson_log:.4f}, p={p_pearson_log:.4g}")

Pearson (raw distance):   r=-0.0424, p=0.1574
Spearman (raw distance):  rho=-0.0280, p=0.3507
Pearson (log distance):   r=-0.1046, p=0.0004757


### Does Competition Distance Relate to Sales?

**Question:** Do stores with closer competitors sell less?

**Method:** Correlated per-store average sales against CompetitionDistance, using Pearson and Spearman on raw distance, plus Pearson on log-transformed distance (since distance is heavily right-skewed).

**Result:** No relationship was detectable on raw distance (p=0.16, p=0.35). After log-transforming distance, a small negative relationship emerged and was statistically significant (r=-0.10, p<0.001).

**Takeaway:** There may be a real but weak effect — closer competitors are mildly associated with lower sales — but it only appears once the skew is corrected for, and it explains very little of the variation in sales on its own. Not yet checked for confounding by StoreType/Assortment.

In [27]:
distance_by_type = (store_level_clean.groupby('StoreType')['CompetitionDistance'].agg(['count', 'mean', 'median', 'std']).round(1))

distance_by_type

,count,mean,median,std
StoreType,,,,
a,601,5123.1,1790.0,8420.9
b,17,1060.6,900.0,831.8
c,148,3522.6,1660.0,5944.8
d,346,6913.1,5040.0,6769.4


In [28]:
for st in sorted(store_level_clean['StoreType'].unique()):
    subset = store_level_clean[store_level_clean['StoreType'] == st]

    if len(subset) >= 15:
        r, p = stats.pearsonr(subset['log_CompetitionDistance'], subset['avg_sales'])
        print(f"StoreType {st}: n={len(subset)}, r={r:.4f}, p={p:.4g}")

    else:
        print(f"StoreType {st}: n={len(subset)} (too small to test reliably)")

StoreType a: n=601, r=-0.1335, p=0.001035
StoreType b: n=17, r=-0.2137, p=0.4101
StoreType c: n=148, r=-0.0734, p=0.3753
StoreType d: n=346, r=0.0508, p=0.3457


### Competition Distance by StoreType

**Method:** Repeated the log-distance versus average-sales correlation separately for each StoreType.

**Result:** A small negative relationship appears only for StoreType a (r=-0.13, p=0.001). No clear relationship was detected for StoreTypes c or d. StoreType b has too few stores (17) for a reliable conclusion.

**Takeaway:** Competition distance is not a consistent sales driver across all store formats. The weak overall relationship is mainly associated with StoreType a and may still reflect other location-related factors.

In [29]:
unknown_distance_stores = (store_level[store_level['CompetitionDistance'].isna()].sort_values('Store'))

unknown_distance_stores

,Store,avg_sales,median_sales,StoreType,Assortment,CompetitionDistance
290,291,8023.039744,7777.5,d,a,NaN
621,622,4317.961735,4253.5,a,c,NaN
878,879,3762.983923,3585.5,d,a,NaN


In [30]:
unknown_store_profiles = (unknown_distance_stores[['Store', 'StoreType', 'Assortment', 'avg_sales']].rename(columns={'avg_sales': 'unknown_store_avg_sales'}))

unknown_store_profiles

,Store,StoreType,Assortment,unknown_store_avg_sales
290,291,d,a,8023.039744
621,622,a,c,4317.961735
878,879,d,a,3762.983923


In [35]:
comparison_by_segment = (
    store_level_clean
    .groupby(['StoreType', 'Assortment'])['avg_sales']
    .agg(['count', 'mean', 'median', 'std'])
    .round(2)
)

comparison_by_segment

count      mean    median      std
StoreType Assortment                                    
a         a             381   6526.68   6046.87  2361.49
          c             220   7592.59   7161.84  2818.11
b         a               7  10987.85  10828.40  5012.54
          b               9   8555.42   7687.46  3500.11
          c               1  17969.56  17969.56      NaN
c         a              77   6831.50   6728.37  1758.16
          c              71   7010.14   6271.13  2559.19
d         a             126   6432.74   6314.12  1533.73
          c             220   7056.14   6853.91  1866.53

### Stores With Unknown Competition Distance

**Question:** Are the three stores with missing CompetitionDistance similar to their peer groups?

**Method:** Compared their average sales with stores in the same StoreType and Assortment segment.

**Result:** Stores 291 and 879 are both StoreType d / Assortment a, but one is above and one is well below the segment's typical sales level. Store 622 belongs to StoreType a / Assortment c and also has below-typical sales.

**Takeaway:** The three unknown-distance stores are not a single consistent group. StoreType and Assortment can inform a future imputation estimate, but cannot recover the true distance. Keep missingness explicit in later modeling.